# **Customer Churn Prediction**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/Customer-Churn-Prediction/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [The Problem Statement](#The-Problem-Statement)
  - [Customers Leave Quietly](#Customers-Leave-Quietly)
  - [Why learn the rules instead of writing them](#Why-learn-the-rules-instead-of-writing-them)
  - [Data: Where these rows actually come from](#Data:-Where-these-rows-actually-come-from)
  - [Data: Training data and serving data are not the same data](#Data:-Training-data-and-serving-data-are-not-the-same-data)
- [The Lifecycle](#The-Lifecycle)
  - [Stage 01: Data Collection](#Stage-01:-Data-Collection)
  - [Stage 02: Data Exploration](#Stage-02:-Data-Exploration)
  - [Stage 03: Preprocessing](#Stage-03:-Preprocessing)
  - [Stage 04: Feature Engineering](#Stage-04:-Feature-Engineering)
  - [Stage 05: Train, Validation, Test Split](#Stage-05:-Train,-Validation,-Test-Split)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/Customer-Churn-Prediction/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/Customer-Churn-Prediction/main/assets/rainbow-divider.png)

# The Problem Statement

## Customers Leave Quietly

**What happens:**

* **A telecom company loses roughly a quarter of its subscribers every year.** Nobody announces they are leaving, they simply stop renewing, and you find out from the billing run weeks later.

    **Churn:** A customer ending their relationship with a service. The opposite of retention.

**Why it is necessary:**

* **Winning a new customer costs far more than keeping one you already have.**

    Keeping an existing customer is far cheaper than acquiring a replacement, so a call placed *before* they leave is worth real money. This is why churn is worth modelling at all. The value is not in predicting the future accurately, it is in placing a cheap intervention before an expensive loss.

**What breaks without it:**

* **Nobody announces they are leaving. They just stop renewing.**

    There is no event to react to. By the time the billing system shows the account closed, the decision was made weeks earlier and the customer is already gone. With no way to tell who is at risk you have two bad options: phone everyone, which burns the entire retention margin on people who were staying anyway, or phone nobody.

**In a real system:**

* **You cannot afford to call everyone, so the question is who.**

    The business constraint is almost always a budget: the retention team can only work a fixed number of accounts a month. That number, not the model, decides how many customers you can act on. So the job is not "predict churn" in the abstract. It is **decide which accounts get this month's calls**, which means we need customers *ranked* by risk, not just labelled.

## Why learn the rules instead of writing them

**What happens:**

* **A hand-written rule is readable, instant, and completely honest about what it does.**

    You could write the rules by hand: month-to-month contract, high monthly bill, short tenure, flag it. That is a perfectly reasonable first system and you should try it. If "month-to-month contract and paying over eighty" catches most of your churners, you have solved the problem without a model, and you can explain it to the retention team in one sentence.

**Why it is necessary:**

* **Rules fail quietly. Behaviour shifts, and the rule keeps firing on last year's pattern.**

    A competitor launches a cheaper fibre plan and the customers at risk change overnight. The rule does not know that. Rules fail quietly. It keeps flagging the same accounts based on last year's pattern and its hit rate quietly decays. That is when learning the rule beats writing it.

* **ML fits when signals are many, weak, interacting, and moving.**

    Machine learning earns its place when the pattern involves many weak signals interacting, and when that pattern drifts. A model refits from data; a rule only changes when someone edits it. Churn is exactly that: contract type matters more for new customers, price matters more for long-tenured ones, and the balance changes each quarter. A model can hold that; a rule list cannot.

**What breaks without it:**

* **If a two line rule really works, ship the rule instead.**

    Reaching for ML when a two-line rule would do leaves you with a system nobody can explain, that needs retraining, monitoring and a serving stack, to buy a point of accuracy.

**In a real system:**

* **Most mature teams run both: a rule-based safety net for the obvious cases, and a model for the rest.** The honest test is whether the model beats the rule on a held-out month.

## Data: Where these rows actually come from

**What happens:**

* **One row per subscriber, snapshotted from the billing system on one day.**

    That is the shape supervised learning needs: each row pairs a situation with what actually happened, so the model has examples to learn the mapping from. The [Telco Customer Churn dataset](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) (7,043 rows x 21 columns, IBM sample data): 7,043 subscribers, one row each, pulled as a snapshot from a carrier's billing and account systems at a single point in time.

**Why it is necessary:**

* **Each row holds account facts, the services they pay for, and the outcome.**

    One row per customer is what makes this a supervised learning problem: every row carries both the situation and the outcome, so a model can be shown examples of both. Tenure, contract type and charges describe the commercial relationship. Internet, security and streaming describe what they use. Together they are the only evidence the model will ever get.

**What breaks without it:**

* **The label is a decision somebody made, not a fact that fell out of the data.**

    Notice that somebody decided what account closed means. Churn here means the account was closed within the observation window. Change that window to 90 days and every label, and therefore every conclusion, changes with it. The label is a choice, not a fact you found.
    
    We cannot treat a snapshot as if it were history. These are the facts as of the export date, not a timeline, so you cannot ask "what changed in the three months before they left".

**In a real system:**

* A real feature table is assembled by joining billing, CRM, usage and support systems on a customer key, usually as a nightly batch job. **The tidy CSV is the end of a pipeline, not the start.**

## Data: Training data and serving data are not the same data

**What happens**

* **Training data is built offline: a warehouse join over months of history.**

    Training data is assembled offline and at leisure: a warehouse query joins the billing database, the CRM, support tickets and usage logs, keyed on the customer, into one wide table, with the outcome already known. Slow queries are fine here. Nobody is waiting.

**Why it is necessary:**

* **Serving is one customer, right now, from live APIs and event streams.**

    Serving works under opposite constraints. One customer, right now, in milliseconds, from live systems: a REST call to the account API, a Kafka event stream of recent usage, a feature store lookup. All of this inside the time budget of a web request, while somebody waits for the answer. The outcome is precisely what you do not have.

**What breaks without it:**

* **Where the two paths disagree, your model quietly stops working.**

    If those two paths compute anything differently, the model quietly stops working. Any feature that is easy offline and hard online will silently differ between the two. Average monthly spend computed over a full billing history offline, but over the last thirty days online, is not the same feature. The model was never trained on what it is now being asked. That difference, **training-serving skew**, is the single most common way a good model fails in production.

**In a real system:**

* **The usual fix is a feature store.**

    Compute each feature once, write it to both an offline table for training and an online store for serving, so both paths read the same definition.

    **Feature store:** A service that computes each feature once and serves it to both training and production, so the definitions cannot drift apart.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/Customer-Churn-Prediction/main/assets/rainbow-divider.png)

# The Lifecycle

**11 stages sit between raw data and a model running in production.**

Everything from collecting data to keeping the model alive breaks into 11 stages. We will walk through all of the 11 stages before writing any code, because the code is really just these stages typed out.

Every project walks them, roughly in this order. For each one: what happens, why it is necessary, what breaks when somebody skips it, and what happens in a real system.

## Stage 01: Data Collection

**What happens:**

* **First decide what one row means, and over what window.**

    Decide what a row is, over what window, and assemble it from the source systems. Here: one row per subscriber, labelled by whether the account closed inside the observation window. That single decision fixes what question the model is able to answer.

**Why it is necessary:**

* Every later choice inherits this one. The unit of analysis, the time window and the label definition together decide what question your model can answer at all.

**What breaks without it:**

* **Let the label window overlap the features and you have built a time machine.**

    Collect the label from a period that overlaps your features and you have built a time machine: the model learns from information that did not exist when the prediction would have been made. It scores beautifully in testing and is useless the day you deploy it.

    **Data Leakage:** Information reaching the model that would not be available at prediction time in the real world.

**In a real system:**

* Usually a scheduled batch job writing a versioned snapshot, so a model trained last month can be reproduced exactly.

## Stage 02: Data Exploration

**What happens:**

* **Look at the data before you touch it. Especially the balance of the outcome.**

    Look before you touch. Shapes, types, missing values, ranges, and above all the balance of the outcome you are trying to predict; how many customers actually churned. The single most consequential fact you will find here is that only about a quarter of these customers churned.

    **Class imbalance:** One outcome far outnumbers the other, so predicting the common one always looks accurate.

**Why it is necessary:**

* **Find the imbalance after training and it has already shaped your conclusions.**

    Exploration is where you find the facts that dictate every later decision: the 26.5% churn rate is what makes accuracy a misleading metric for the rest of this project. Miss that now and you will have optimised, compared models and reported a number, all against a metric that quietly rewarded agreeing with the majority; and you will never notice it. The work is not wrong so much as meaningless.

**What breaks without it:**

* Skip it and you discover the wrong dtype, the impossible value or the class imbalance after training, when it has already quietly shaped your results.

**In a real system:**

* Automated data-quality checks run on every batch: schema, null rates, and distribution ranges, failing the pipeline rather than the model.

## Stage 03: Preprocessing

**What happens:**

* **Make it machine-readable without changing what it means.**

Make the data machine-readable without changing what it means: fix types, fill or mark missing values, and turn categories into numbers. Every conversion is a decision you are making on the model's behalf.

**Why it is necessary:**

* **Number your categories 0, 1, 2 and you have invented an order that is not there.**

    Models consume numeric arrays. Everything human-readable in the raw table has to become a number, and every one of those conversions is a decision you are making on the model's behalf.

    The classic mistake is numbering categories 0, 1, 2, etc. Now the model believes a two-year contract is twice a one-year contract, and you invented that ordering yourself.

    **Encoding:** Turning a category into numbers a model can use, without implying an order that is not there.

**What breaks without it**

* Encoding categories as 0, 1, 2 invents an ordering that does not exist. Filling missing values with a mean invents data. Both are silent.

**In a real system**

* These steps must be saved as fitted objects and shipped with the model, not re-implemented in the serving code.

## Stage 04: Feature Engineering

**What happens:**

* **Build the columns that make the pattern easy for the model to see.**

    Build the columns that make the pattern easy to see: ratios, differences, flags and aggregates derived from what you already have.

    **Feature Engineering:** Building new inputs from existing ones so the pattern is easier for a model to find.

**Why it is necessary:**

* A model can only find structure that is reachable through its own machinery. Handing it a ratio saves it from approximating that ratio with a staircase of splits. Handing it the right shape directly is often worth more than a better algorithm.

**What breaks without it:**

* **Build a feature from something you will not have at serving time and that is leakage.**

    Just make sure you have the ingredients at prediction time. Engineer a feature from information that will not exist at serving time, and you have built leakage by hand. Build a feature from something you only learn afterwards and that is leakage, and it looks perfectly reasonable right up until production. It is the **most common self-inflicted wound in applied ML**, because the feature looks legitimate, improves your score, and only fails once it reaches production.

**In a real system**

* Feature definitions live in one place and are versioned, because a redefined feature invalidates every model trained on the old one.

## Stage 05: Train, Validation, Test Split

**What happens:**

* **Three cuts: learn from one, choose with one, be judged on one.**

    Cut the data three ways before anything is fitted. `Train set` to learn from, `validation set` to make choices with (compare models and settings against each other), `test set` to be judged on exactly once (test set stays sealed until every decision has already been made).

    **Validation set:** Data used to compare models and settings. It is spent by those comparisons, which is why the test set is kept separate.

**Why it is necessary:**

* **Every choice you make on a set burns it as an honest estimate.**

    Every look you take at a set spends it. Every decision you make using a set of data burns that set as an honest estimate. The test set stays sealed so that at the end you have one number you can trust.
    
    Choose your model on the test set and its score stops telling you how well you generalize and starts measuring how hard you searched. The number stays high while the real performance falls.

**What breaks without it:**

* Choose your model on the test set and its score becomes just another training score. You will ship something worse than you believe, with no way to know.

**In a real system:**

* For anything time-dependent, split by time rather than at random, because predicting the past from the future is the easiest accidental leak there is.

![rainbow](https://raw.githubusercontent.com/ancilcleetus/Customer-Churn-Prediction/main/assets/rainbow-divider.png)

# References

- **[AI-ML Companion](https://aimlcompanion.ai/)**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/Customer-Churn-Prediction/main/assets/rainbow-divider.png)